# Chapter 07: Weight Quantization (GPTQ, AWQ, GGUF)

**Goal:** Understand how reducing weight precision directly improves decode throughput — and the quality tradeoffs.

From Chapter 1: decode arithmetic intensity is ~1 FLOP/byte (weight-dominated).
If we halve the bytes per weight (FP16 -> INT8), intensity doubles (2 F/B).
At INT4, intensity quadruples (4 F/B). The same compute with fewer bytes = faster.

**Model:** LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers)

**Approach:** Fill in every `# YOUR CODE` section yourself — that's the learning.

**Hardware:** A100 80GB SXM4
- Measured HBM bandwidth: 1774 GB/s
- FP16 TFLOPS: 237
- Ridge point: ~134 FLOPS/byte

## 1. Why Quantization Matters at Batch=1

Recall from Chapter 1: during decode, every linear layer does a matrix-vector multiply.
The weight matrix is read once, the input vector is tiny.

**Intensity at decode (batch=1, single token):**
- `y = Wx`: FLOPs = `2 * M * N`, bytes dominated by weight `M * N * bytes_per_weight`
- FP16: 2 bytes/weight -> intensity ~ 2 FLOP / 2 bytes = **1 FLOP/byte**
- INT8: 1 byte/weight -> intensity ~ 2 FLOP / 1 byte = **2 FLOPS/byte**
- INT4: 0.5 bytes/weight -> intensity ~ 2 FLOP / 0.5 bytes = **4 FLOPS/byte**

All still far below the ridge (~134 F/B), but each halving of weight size = **2x theoretical speedup**.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda')
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- YOUR CODE: calculate decode intensity at FP16, INT8, INT4 ---
# Hint: intensity = 2 * M * N / (M * N * bytes_per_weight) = 2 / bytes_per_weight

peak_bw = 1774   # GB/s
peak_tfl = 237    # TFLOPS
ridge_point = peak_tfl * 1000 / peak_bw  # ~134 FLOPS/byte

# Decode: y = Wx where W is (N, M), x is (M, 1)
M, N = 2048, 2048  # Example: Q projection

formats = {
    'FP16': 2.0,   # 2 bytes per weight
    'INT8': 1.0,   # 1 byte per weight
    'INT4': 0.5,   # 0.5 bytes per weight
}

print(f"Decode intensity for linear layer ({M}x{N}):")
print(f"{'Format':<8} {'Bytes/W':>8} {'Weight MB':>10} {'Intensity':>12} {'Bound':>15}")
print("-" * 60)

intensities = {}
for fmt, bpw in formats.items():
    flops = 2 * M * N
    weight_bytes = M * N * bpw
    input_bytes = M * 2  # FP16 input vector
    output_bytes = N * 2  # FP16 output vector
    total_bytes = weight_bytes + input_bytes + output_bytes
    intensity = flops / total_bytes  # YOUR CODE
    intensities[fmt] = intensity
    bound = 'Memory-bound' if intensity < ridge_point else 'Compute-bound'
    print(f"{fmt:<8} {bpw:>8.1f} {weight_bytes/1e6:>10.2f} {intensity:>10.1f} F/B {bound:>15}")

print(f"\nRidge point: {ridge_point:.0f} FLOPS/byte")
print(f"All formats are memory-bound at batch=1, but INT4 is {intensities['INT4']/intensities['FP16']:.0f}x more efficient.")

In [ ]:
# --- YOUR CODE: plot FP16, INT8, INT4 decode intensity on roofline ---

intensity_range = np.logspace(-1, 3, 1000)
memory_bound = peak_bw / 1000 * intensity_range
roofline = np.minimum(memory_bound, peak_tfl)

fig, ax = plt.subplots(figsize=(12, 7))
ax.loglog(intensity_range, roofline * 1000, 'k-', linewidth=2.5, label='A100 Roofline', zorder=10)

mem_mask = intensity_range < ridge_point
ax.fill_between(intensity_range[mem_mask], roofline[mem_mask] * 1000,
                alpha=0.15, color='blue', label='Memory-bound region')
ax.fill_between(intensity_range[~mem_mask], roofline[~mem_mask] * 1000,
                alpha=0.15, color='red', label='Compute-bound region')
ax.axvline(ridge_point, color='red', linestyle='--', linewidth=1, alpha=0.5)

colors_fmt = {'FP16': '#d62728', 'INT8': '#ff7f0e', 'INT4': '#2ca02c'}
markers_fmt = {'FP16': 'o', 'INT8': 's', 'INT4': '^'}

for fmt, intensity in intensities.items():
    achieved = peak_bw * intensity * 0.7  # memory-bound estimate with ~70% efficiency
    ax.loglog(intensity, achieved, marker=markers_fmt[fmt], markersize=14,
             color=colors_fmt[fmt], label=f'{fmt} decode ({intensity:.1f} F/B)', zorder=5)

# Arrow from FP16 to INT4
ax.annotate('', xy=(intensities['INT4'], peak_bw * intensities['INT4'] * 0.7),
            xytext=(intensities['FP16'], peak_bw * intensities['FP16'] * 0.7),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text(2.0, peak_bw * 0.4, 'Quantization moves\nops rightward', fontsize=10, color='purple', ha='center')

ax.set_xlabel('Arithmetic Intensity (FLOPS/byte)', fontsize=12, fontweight='bold')
ax.set_ylabel('Performance (GFLOPS)', fontsize=12, fontweight='bold')
ax.set_title('Weight Quantization: Decode Intensity on A100 Roofline', fontsize=14, fontweight='bold')
ax.grid(True, which='both', alpha=0.3, linestyle=':')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0.1, 1000)
ax.set_ylim(10, 500000)
plt.tight_layout()
plt.show()

## 2. How Quantization Works

**Linear quantization** maps floating-point values to integers:

```
Quantize:   x_q = round(x / scale + zero_point)
Dequantize: x_approx = (x_q - zero_point) * scale
```

Where:
- `scale = (max_val - min_val) / (2^bits - 1)`
- `zero_point` maps the real-valued 0 to an integer

The key question: **how much error does this introduce?**

In [ ]:
# --- YOUR CODE: implement quantize/dequantize for a weight matrix ---

def quantize(x, bits=8):
    """Symmetric quantization: no zero point, scale based on max abs value."""
    # YOUR CODE
    qmax = 2 ** (bits - 1) - 1  # Hint: 127 for INT8, 7 for INT4
    qmin = -(2 ** (bits - 1))   # Hint: -128 for INT8, -8 for INT4
    scale = x.abs().max() / qmax  # Hint: scale = max(|x|) / qmax
    x_q = torch.clamp(torch.round(x / scale), qmin, qmax).to(torch.int8)  # Hint: round and clamp
    return x_q, scale

def dequantize(x_q, scale):
    """Recover approximate float values."""
    # YOUR CODE
    return x_q.float() * scale  # Hint: x_q * scale

# Test on a real weight matrix
W = torch.randn(2048, 2048, dtype=torch.float16, device=device)

for bits in [8, 4]:
    W_q, scale = quantize(W.float(), bits=bits)
    W_approx = dequantize(W_q, scale).half()

    # --- YOUR CODE: measure reconstruction error ---
    mse = ((W - W_approx) ** 2).mean().item()  # Hint: mean squared error
    max_err = (W - W_approx).abs().max().item()  # Hint: max absolute error
    snr = 10 * np.log10((W ** 2).mean().item() / mse)  # Hint: signal-to-noise ratio in dB

    print(f"INT{bits} quantization:")
    print(f"  Scale: {scale:.6f}")
    print(f"  MSE: {mse:.6f}")
    print(f"  Max error: {max_err:.6f}")
    print(f"  SNR: {snr:.1f} dB")
    print(f"  Memory: {W.numel() * 2 / 1e6:.1f} MB (FP16) -> {W.numel() * bits / 8 / 1e6:.1f} MB (INT{bits})")
    print()

## 3. Weight-Only Quantization

**Key idea:** Store weights in INT4/INT8 on disk and in HBM, but dequantize to FP16 just before the matmul.

Benefits:
- Smaller model on disk and in VRAM
- Less HBM bandwidth needed (main bottleneck at decode)
- FP16 compute path is unchanged (same tensor core utilization)

The dequantization cost is tiny vs. the bandwidth savings.

In [ ]:
# --- YOUR CODE: implement weight-only quantized linear layer ---

class QuantizedLinear(nn.Module):
    def __init__(self, in_features, out_features, bits=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bits = bits

        # Placeholder — will be filled by quantize_from()
        self.register_buffer('weight_q', torch.zeros(out_features, in_features, dtype=torch.int8))
        self.register_buffer('scale', torch.zeros(1))

    def quantize_from(self, linear_layer):
        """Quantize a nn.Linear layer's weights."""
        W = linear_layer.weight.data.float()
        # YOUR CODE
        self.weight_q, self.scale = quantize(W, bits=self.bits)  # Hint: use quantize()
        self.weight_q = self.weight_q.to(linear_layer.weight.device)
        self.scale = self.scale.to(linear_layer.weight.device)

    def forward(self, x):
        # YOUR CODE: dequantize weight and do matmul
        W_fp16 = (self.weight_q.float() * self.scale).half()  # Hint: dequantize to FP16
        return torch.nn.functional.linear(x, W_fp16)  # Hint: F.linear(x, W)

# Benchmark
linear_fp16 = nn.Linear(2048, 2048, bias=False, dtype=torch.float16, device=device)

q_linear_int8 = QuantizedLinear(2048, 2048, bits=8).to(device)
q_linear_int8.quantize_from(linear_fp16)

q_linear_int4 = QuantizedLinear(2048, 2048, bits=4).to(device)
q_linear_int4.quantize_from(linear_fp16)

x = torch.randn(1, 1, 2048, dtype=torch.float16, device=device)  # Decode: single token

def cuda_timer(fn, warmup=10, repeats=100):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats): fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats

# --- YOUR CODE: benchmark FP16 vs INT8 vs INT4 ---
with torch.no_grad():
    t_fp16 = cuda_timer(lambda: linear_fp16(x))
    t_int8 = cuda_timer(lambda: q_linear_int8(x))
    t_int4 = cuda_timer(lambda: q_linear_int4(x))

# Memory comparison
mem_fp16 = 2048 * 2048 * 2  # FP16 weight bytes
mem_int8 = 2048 * 2048 * 1 + 4  # INT8 weight + scale
mem_int4 = 2048 * 2048 // 2 + 4  # INT4 weight + scale

print(f"{'Format':<8} {'Time (ms)':>10} {'Memory (MB)':>12} {'Speedup':>10}")
print("-" * 45)
print(f"{'FP16':<8} {t_fp16:>10.4f} {mem_fp16/1e6:>12.2f} {'1.00x':>10}")
print(f"{'INT8':<8} {t_int8:>10.4f} {mem_int8/1e6:>12.2f} {t_fp16/t_int8:>9.2f}x")
print(f"{'INT4':<8} {t_int4:>10.4f} {mem_int4/1e6:>12.2f} {t_fp16/t_int4:>9.2f}x")
print(f"\nNote: our naive dequantize-then-matmul is slow. Real INT4 kernels (GPTQ/AWQ)")
print(f"fuse dequantization into the matmul kernel for much better performance.")

## 4. GPTQ: Post-Training Quantization

**GPTQ** (Generative Pre-trained Transformer Quantization) is based on Optimal Brain Quantization:

1. Quantize one weight at a time (column by column)
2. After quantizing each weight, **adjust remaining weights** to compensate for the error
3. Use the inverse Hessian (from calibration data) to find optimal adjustments

This is much better than naive round-to-nearest because later weights compensate for earlier rounding errors.

**Typical setup:** INT4 weights with group_size=128 (separate scale per 128 weights).

In [ ]:
# --- YOUR CODE: quantize LLaMA-1B with GPTQ using auto_gptq ---
# Note: this cell requires auto_gptq installed: pip install auto-gptq
# If not available, we demonstrate the concept with manual group quantization.

# Group quantization: separate scale for each group of weights
def group_quantize(W, bits=4, group_size=128):
    """Quantize with per-group scaling (like GPTQ/AWQ)."""
    # YOUR CODE
    rows, cols = W.shape
    assert cols % group_size == 0, f"cols ({cols}) must be divisible by group_size ({group_size})"

    W_grouped = W.reshape(rows, -1, group_size)  # Hint: reshape into groups
    qmax = 2 ** (bits - 1) - 1
    scales = W_grouped.abs().amax(dim=-1, keepdim=True) / qmax  # Hint: per-group scale
    scales = scales.clamp(min=1e-8)
    W_q = torch.clamp(torch.round(W_grouped / scales), -qmax - 1, qmax)
    W_deq = (W_q * scales).reshape(rows, cols)

    return W_q.to(torch.int8), scales, W_deq

# Compare naive (per-tensor) vs group quantization
W = torch.randn(2048, 2048, dtype=torch.float32, device=device)

print(f"INT4 Quantization Comparison:")
print(f"{'Method':<25} {'MSE':>12} {'SNR (dB)':>10} {'Scale overhead':>15}")
print("-" * 65)

# Per-tensor (naive)
W_q_naive, scale_naive = quantize(W, bits=4)
W_deq_naive = dequantize(W_q_naive, scale_naive)
mse_naive = ((W - W_deq_naive) ** 2).mean().item()
snr_naive = 10 * np.log10((W ** 2).mean().item() / mse_naive)
print(f"{'Per-tensor (naive)':<25} {mse_naive:>12.6f} {snr_naive:>10.1f} {'4 bytes':>15}")

# Per-group (GPTQ-style)
for gs in [128, 64, 32]:
    _, scales_g, W_deq_g = group_quantize(W, bits=4, group_size=gs)
    mse_g = ((W - W_deq_g) ** 2).mean().item()
    snr_g = 10 * np.log10((W ** 2).mean().item() / mse_g)
    scale_bytes = scales_g.numel() * 2  # FP16 scales
    print(f"{'Group size=' + str(gs):<25} {mse_g:>12.6f} {snr_g:>10.1f} {str(scale_bytes) + ' bytes':>15}")

print(f"\nSmaller groups = better accuracy but more scale overhead.")
print(f"group_size=128 is the standard GPTQ default (good balance).")

In [ ]:
# Using auto_gptq library (if installed)
try:
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

    # --- YOUR CODE: set up GPTQ quantization config ---
    quantize_config = BaseQuantizeConfig(
        bits=4,               # YOUR CODE: 4-bit quantization
        group_size=128,       # YOUR CODE: group size 128
        desc_act=False,       # YOUR CODE: disable activation reordering for speed
    )

    print(f"GPTQ config: bits={quantize_config.bits}, group_size={quantize_config.group_size}")
    print(f"\nTo quantize (takes ~5-10 min on A100):")
    print(f"  model = AutoGPTQForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B', quantize_config)")
    print(f"  model.quantize(calibration_dataset)")
    print(f"  model.save_quantized('llama-1b-gptq-int4')")

except ImportError:
    print("auto_gptq not installed. Install with: pip install auto-gptq")
    print("\nGPTQ quantization concept:")
    print("  1. Run calibration data through model to collect Hessian info")
    print("  2. Quantize weights column-by-column")
    print("  3. Adjust remaining weights to minimize output error")
    print("  4. Result: INT4 weights with minimal perplexity loss")

## 5. AWQ: Activation-Aware Quantization

**AWQ** (Activation-Aware Weight Quantization) key insight:

**Not all weights are equally important.** Some weights consistently produce high-magnitude activations.
Quantization error on these "salient" weights causes disproportionate output error.

AWQ approach:
1. Run calibration data, identify weights with high activation magnitudes
2. Scale salient weight channels UP before quantization (preserves their precision)
3. Scale the corresponding activations DOWN to compensate
4. Net effect: zero math change, but salient weights get more INT4 levels

In [ ]:
# --- YOUR CODE: demonstrate AWQ's core idea — activation-aware scaling ---

# Simulate: some channels are much more important than others
W = torch.randn(2048, 2048, dtype=torch.float32, device=device)
x_calib = torch.randn(128, 2048, dtype=torch.float32, device=device)  # Calibration data

# Measure activation magnitude per channel
# YOUR CODE: compute per-channel activation importance
act_magnitudes = x_calib.abs().mean(dim=0)  # Hint: mean |activation| per input channel

# Identify salient channels (top 1%)
threshold = torch.quantile(act_magnitudes, 0.99)
salient_mask = act_magnitudes > threshold
n_salient = salient_mask.sum().item()
print(f"Salient channels (top 1%): {n_salient} out of 2048")
print(f"Mean activation - salient: {act_magnitudes[salient_mask].mean():.4f}")
print(f"Mean activation - other:   {act_magnitudes[~salient_mask].mean():.4f}")

# Naive INT4 quantization
_, _, W_deq_naive = group_quantize(W, bits=4, group_size=128)

# AWQ-style: scale salient channels before quantization
# YOUR CODE: create per-channel scale that's larger for salient channels
awq_scale = torch.ones(2048, device=device)
awq_scale[salient_mask] = act_magnitudes[salient_mask] / act_magnitudes[salient_mask].min()  # Hint: scale up salient

# Scale weights, quantize, scale back
W_scaled = W * awq_scale.unsqueeze(0)  # Hint: W * scale per input channel
_, _, W_deq_awq_scaled = group_quantize(W_scaled, bits=4, group_size=128)
W_deq_awq = W_deq_awq_scaled / awq_scale.unsqueeze(0)  # Hint: undo the scaling

# Compare errors
mse_naive = ((W - W_deq_naive) ** 2).mean().item()
mse_awq = ((W - W_deq_awq) ** 2).mean().item()

# Error on salient channels specifically
mse_naive_salient = ((W[:, salient_mask] - W_deq_naive[:, salient_mask]) ** 2).mean().item()
mse_awq_salient = ((W[:, salient_mask] - W_deq_awq[:, salient_mask]) ** 2).mean().item()

print(f"\n{'Metric':<30} {'Naive INT4':>12} {'AWQ INT4':>12}")
print("-" * 55)
print(f"{'Overall MSE':<30} {mse_naive:>12.6f} {mse_awq:>12.6f}")
print(f"{'Salient channel MSE':<30} {mse_naive_salient:>12.6f} {mse_awq_salient:>12.6f}")
print(f"{'Salient improvement':<30} {'':>12} {mse_naive_salient/mse_awq_salient:>11.1f}x")

In [ ]:
# Using autoawq library (if installed)
try:
    from awq import AutoAWQForCausalLM

    # --- YOUR CODE: quantize with AWQ ---
    print(f"AWQ quantization:")
    print(f"  model = AutoAWQForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B')")
    print(f"  model.quantize(tokenizer, quant_config={{'zero_point': True, 'q_group_size': 128, 'w_bit': 4}})")
    print(f"  model.save_quantized('llama-1b-awq-int4')")

except ImportError:
    print("autoawq not installed. Install with: pip install autoawq")
    print("\nAWQ vs GPTQ comparison:")
    print("  - GPTQ: optimal per-weight rounding, slower quantization")
    print("  - AWQ: activation-aware channel scaling, better for INT4 on newer models")
    print("  - Both use group_size=128 for INT4")
    print("  - AWQ generally better perplexity at 4-bit for LLaMA-family models")

## 6. GGUF and llama.cpp

**GGUF** is the file format for llama.cpp — optimized for CPU and hybrid CPU/GPU inference.

Key quantization types:

| Type | Bits/weight | Block size | Quality | Use case |
|------|------------|------------|---------|----------|
| Q4_0 | 4.5 | 32 | Low | Maximum compression |
| Q4_K_M | 4.83 | 256 | Medium | Good balance |
| Q5_K_M | 5.68 | 256 | High | Quality-sensitive |
| Q8_0 | 8.5 | 32 | Very high | Near-lossless |
| F16 | 16.0 | - | Baseline | Reference |

"K" variants use k-quants: different bit widths for important vs unimportant layers.

The block sizes add scale/offset overhead, so effective bits > nominal bits.

In [ ]:
# --- YOUR CODE: understand GGUF quantization table ---

# LLaMA-3.2-1B total parameters
total_params = sum(p.numel() for p in model.parameters()) if 'model' in dir() else 1_235_814_400

gguf_types = [
    ('F16',    16.0,  'Baseline — full precision'),
    ('Q8_0',    8.5,  'Near-lossless, ~0.1 perplexity increase'),
    ('Q5_K_M',  5.68, 'High quality, recommended for quality-sensitive use'),
    ('Q4_K_M',  4.83, 'Good balance of size and quality'),
    ('Q4_0',    4.5,  'Maximum compression, noticeable quality loss'),
]

print(f"LLaMA-3.2-1B: {total_params / 1e9:.2f}B parameters")
print(f"\n{'Type':<10} {'Bits/W':>7} {'Size (GB)':>10} {'Ratio':>7} {'Notes'}")
print("-" * 70)

for name, bpw, notes in gguf_types:
    size_gb = total_params * bpw / 8 / 1e9
    ratio = 16.0 / bpw
    # YOUR CODE: calculate model size and compression ratio
    print(f"{name:<10} {bpw:>7.2f} {size_gb:>10.2f} {ratio:>6.1f}x  {notes}")

print(f"\nMemory saved going from F16 -> Q4_K_M:")
fp16_gb = total_params * 16 / 8 / 1e9
q4km_gb = total_params * 4.83 / 8 / 1e9
print(f"  {fp16_gb:.2f} GB -> {q4km_gb:.2f} GB ({fp16_gb - q4km_gb:.2f} GB saved, {fp16_gb/q4km_gb:.1f}x smaller)")

## 7. Benchmark: FP16 vs INT8 vs INT4

Let's compare real-world performance of quantized models.
We measure:
- **VRAM usage** (model weight memory)
- **Tokens/sec** (prefill and decode)
- **Perplexity** (quality metric — lower is better)

In [ ]:
# --- YOUR CODE: benchmark FP16 vs INT8 vs INT4 models ---
# Note: requires quantized models to be available.
# We demonstrate with our QuantizedLinear for the single-layer comparison.

# Single-layer benchmark: q_proj as representative
from transformers import AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B"

# FP16 model layer
model_fp16 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
model_fp16.eval()
q_proj_fp16 = model_fp16.model.layers[0].self_attn.q_proj

# Create quantized versions of q_proj
q_proj_int8 = QuantizedLinear(2048, 2048, bits=8).to(device)
q_proj_int8.quantize_from(q_proj_fp16)

q_proj_int4 = QuantizedLinear(2048, 2048, bits=4).to(device)
q_proj_int4.quantize_from(q_proj_fp16)

# Decode benchmark (batch=1, seq=1)
x_decode = torch.randn(1, 1, 2048, dtype=torch.float16, device=device)

# Prefill benchmark (batch=1, seq=512)
x_prefill = torch.randn(1, 512, 2048, dtype=torch.float16, device=device)

results = []
with torch.no_grad():
    for name, layer in [('FP16', q_proj_fp16), ('INT8', q_proj_int8), ('INT4', q_proj_int4)]:
        t_decode = cuda_timer(lambda l=layer: l(x_decode))
        t_prefill = cuda_timer(lambda l=layer: l(x_prefill))

        # Estimate memory
        if name == 'FP16':
            mem_mb = 2048 * 2048 * 2 / 1e6
        elif name == 'INT8':
            mem_mb = 2048 * 2048 * 1 / 1e6
        else:
            mem_mb = 2048 * 2048 * 0.5 / 1e6

        results.append({
            'Format': name,
            'Memory (MB)': mem_mb,
            'Decode (ms)': t_decode,
            'Prefill (ms)': t_prefill,
        })

# --- YOUR CODE: create comparison table ---
print(f"q_proj layer benchmark (2048x2048):")
print(f"{'Format':<8} {'Memory':>10} {'Decode':>12} {'Prefill':>12} {'Decode speedup':>15}")
print("-" * 62)
for r in results:
    speedup = results[0]['Decode (ms)'] / r['Decode (ms)']
    print(f"{r['Format']:<8} {r['Memory (MB)']:>8.1f} MB {r['Decode (ms)']:>10.4f} ms {r['Prefill (ms)']:>10.4f} ms {speedup:>13.2f}x")

print(f"\nNote: our naive dequant-then-matmul doesn't show the full benefit.")
print(f"Real GPTQ/AWQ kernels fuse dequant into the matmul for 2-4x decode speedup.")

## 8. When to Use Which

**Decision tree:**

```
Is VRAM the bottleneck?
  YES -> INT4 (GPTQ or AWQ)
    - AWQ: better for LLaMA-family, faster quantization
    - GPTQ: more mature, wider model support
  NO -> Is quality critical?
    YES -> INT8 or FP16
    NO  -> INT4 for speed

Running on CPU?
  YES -> GGUF (llama.cpp)
    - Q4_K_M for most cases
    - Q5_K_M if quality matters
    - Q8_0 if you have the RAM

Serving at scale?
  YES -> AWQ (best INT4 kernel support for GPU serving)
```

**Rule of thumb:** Start with AWQ-INT4. Only go to FP16/INT8 if you measure perplexity degradation.

## 9. Roofline Analysis

Let's put the full picture together: decode linear layer at FP16, INT8, INT4 on the roofline.

Key insight: quantization can't make decode compute-bound (we'd need batch >> 1 for that).
But it makes decode **less** memory-bound, extracting more of the available bandwidth.

In [ ]:
# --- YOUR CODE: plot decode linear layer at FP16, INT8, INT4 on roofline ---

# Full model decode analysis: all 7 linear layers per block, 16 blocks
layers_per_block = [
    ('q_proj',    2048, 2048),
    ('k_proj',    2048, 512),
    ('v_proj',    2048, 512),
    ('o_proj',    2048, 2048),
    ('gate_proj', 2048, 8192),
    ('up_proj',   2048, 8192),
    ('down_proj', 8192, 2048),
]
n_blocks = 16

print(f"Full model decode analysis (batch=1, single token):")
print(f"{'Format':<8} {'Total weights':>15} {'Total FLOPs':>15} {'Intensity':>12} {'Theoretical tok/s':>18}")
print("-" * 75)

for fmt, bpw in [('FP16', 2.0), ('INT8', 1.0), ('INT4', 0.5)]:
    total_flops = 0
    total_bytes = 0
    for name, in_dim, out_dim in layers_per_block:
        flops = 2 * in_dim * out_dim
        weight_bytes = in_dim * out_dim * bpw
        total_flops += flops
        total_bytes += weight_bytes

    total_flops *= n_blocks
    total_bytes *= n_blocks

    # YOUR CODE: calculate intensity and theoretical throughput
    intensity = total_flops / total_bytes  # YOUR CODE
    # Theoretical: time = bytes / bandwidth, tok/s = 1 / time
    time_s = total_bytes / (peak_bw * 1e9)  # YOUR CODE
    tok_per_s = 1.0 / time_s  # YOUR CODE

    print(f"{fmt:<8} {total_bytes/1e6:>13.1f} MB {total_flops/1e9:>13.3f} GF {intensity:>10.1f} F/B {tok_per_s:>16,.0f}")

print(f"\nRidge point: {ridge_point:.0f} FLOPS/byte")
print(f"All formats remain memory-bound at batch=1.")
print(f"INT4 achieves ~4x the theoretical throughput of FP16 by reading 4x fewer bytes.")

In [ ]:
# Full roofline plot with all formats
intensity_range = np.logspace(-1, 3, 1000)
memory_bound = peak_bw / 1000 * intensity_range
roofline = np.minimum(memory_bound, peak_tfl)

fig, ax = plt.subplots(figsize=(12, 8))
ax.loglog(intensity_range, roofline * 1000, 'k-', linewidth=2.5, label='A100 Roofline', zorder=10)

mem_mask = intensity_range < ridge_point
ax.fill_between(intensity_range[mem_mask], roofline[mem_mask] * 1000,
                alpha=0.15, color='blue', label='Memory-bound region')
ax.fill_between(intensity_range[~mem_mask], roofline[~mem_mask] * 1000,
                alpha=0.15, color='red', label='Compute-bound region')
ax.axvline(ridge_point, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.text(ridge_point * 1.1, peak_tfl * 500, f'Ridge: {ridge_point:.0f} F/B', fontsize=10, color='red')

# Plot decode intensity for each format
decode_data = [
    ('FP16 decode',  1.0, '#d62728', 'o'),
    ('INT8 decode',  2.0, '#ff7f0e', 's'),
    ('INT4 decode',  4.0, '#2ca02c', '^'),
]

for label, intensity, color, marker in decode_data:
    achieved = peak_bw * intensity * 0.7  # ~70% bandwidth utilization
    ax.loglog(intensity, achieved, marker=marker, markersize=14,
             color=color, label=f'{label} ({intensity:.0f} F/B)', zorder=5)

# Also show prefill for reference
prefill_intensity = 170  # Approximate for seq=512 FFN layer
ax.loglog(prefill_intensity, peak_tfl * 1000 * 0.8, 'kD', markersize=12,
         label=f'FP16 prefill (~{prefill_intensity} F/B)', zorder=5, alpha=0.5)

ax.set_xlabel('Arithmetic Intensity (FLOPS/byte)', fontsize=12, fontweight='bold')
ax.set_ylabel('Performance (GFLOPS)', fontsize=12, fontweight='bold')
ax.set_title('Weight Quantization: Moving Decode Rightward on the Roofline', fontsize=14, fontweight='bold')
ax.grid(True, which='both', alpha=0.3, linestyle=':')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0.1, 1000)
ax.set_ylim(10, 500000)
plt.tight_layout()
plt.show()

## Revision Notes

*Fill this in after running all sections.*

---

**Decode intensity by format:**
- FP16: ___ FLOPS/byte
- INT8: ___ FLOPS/byte
- INT4: ___ FLOPS/byte
- Ridge point: ___ FLOPS/byte

**Quantization error (INT8 vs INT4):**
- INT8 SNR: ___ dB
- INT4 SNR: ___ dB
- Group quantization (g=128) SNR improvement: ___

**GPTQ vs AWQ:**



**GGUF recommended type:**



**Memory savings (LLaMA-1B):**
- FP16: ___ GB
- INT4: ___ GB

**When to use each format:**



**What surprised me:**



---
*Next: Chapter 8 — FP8, KV Cache Quantization & Mixed Precision*